In [2]:
# ============================================================
# ACLED Source Split (National vs Regional/Local Newspapers) + EDA + EXPORTS
#
# Does EXACTLY what Nausheen asked:
# 1) Using ACLED, divide the data by source of reporting:
#       - national-level newspapers
#       - regional/local newspapers
#    (and keeps non-newspaper sources separate as "other" so they don't contaminate either bucket)
# 2) Basic EDA comparing national vs regional/local:
#       - #events, #fatalities, mean/median fatalities
#       - event-type composition
#       - geographic coverage (Admin1/Admin2/location counts)
#       - state-level (Admin1) differences in coverage
#       - trends over time
# 3) Exports:
#       - classified dataset
#       - all tables (CSV)
#       - all plots (PNG)
#       - a manifest and a ZIP with everything
#
# Copy into a Jupyter notebook and run top-to-bottom.
# ============================================================

import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
ACLED_CSV_PATH = "ACLED Data_2025-12-27.csv"  # <-- change if needed
DATE_MIN = "2016-01-01"
DATE_MAX = "2024-12-31"

OUTDIR = Path("outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) DEFINE NATIONAL NEWSPAPERS (EDIT THIS LIST)
# =========================
# Put ONLY the sources you want treated as "national-level newspapers".
# These must match how ACLED writes them in the 'source' field.
NATIONAL_NEWSPAPERS = {
    "Times of India",
    "The Hindu",
    "Hindustan Times",
    "Indian Express",
    "The Indian Express",
    "The New Indian Express",
    "The Tribune",
    "Telegraph (India)",
    "Statesman (India)",
    "Business Standard (India)",
    "Economic Times (India)",
    "The Economic Times India",
    "Live Mint",
    "Outlook",
    "Outlook (India)",
    "Week (India)",
    "The Print",
    "Scroll (India)",
    "Wire (India)",
    "Quint (India)",
    "Firstpost",
}

# =========================
# 2) DEFINE NON-NEWSPAPER SOURCES (to keep them out of regional/local)
# =========================
# These are agencies/wires, broadcasters, NGOs, international outlets, databases, etc.
# Any event that only has these sources is labeled "other" (not national/regional).
NON_NEWSPAPER_SOURCES = {
    # Agencies/wires
    "Press Trust of India",
    "United News of India",
    "Indo-Asian News Service",
    "Asia News International",
    "Asia News International (ANI)",
    "Reuters",
    "AFP (Agence France-Presse)",
    "AP (Associated Press)",
    "Associated Press of Pakistan",

    # Broadcasters / TV / radio
    "All India Radio",
    "Al Jazeera",
    "BBC News",
    "Deutsche Welle",
    "WION",
    "Times Now (India)",
    "Republic World",
    "News 18 (India)",
    "Zee News (India)",
    "India TV",
    "TV9 Bangla",
    "CNBC",

    # NGOs / monitoring / databases
    "Amnesty International",
    "CIVICUS (NGO)",
    "Committee to Protect Journalists (NGO)",
    "Commonwealth Human Rights Initiative (NGO)",
    "Forum Asia (NGO)",
    "Front Line Defenders (NGO)",
    "Global Witness (NGO)",
    "Scholars at Risk (NGO)",
    "Human Rights Defenders' Alert - India",
    "Aid Worker Security Database",
    "Insecurity Insight",
    "ProtectDefenders.eu",
    "GardaWorld",
    "South Asia Terrorism Portal",

    # International/foreign outlets that appear in your verified list
    "Daily Mail (UK)",
    "Independent (UK)",
    "New York Times",
    "Washington Post",
    "Xinhua",
    "Dawn (Pakistan)",
    "Express Tribune (Pakistan)",
    "Dhaka Tribune",
    "Daily Star (Bangladesh)",
    "Daily Sun (Bangladesh)",
    "Daily Times (Pakistan)",
    "Daily Observer (Bangladesh)",
    "Daily Regional Times (Pakistan)",
    "Nation (Pakistan)",
    "Pakistan Observer",
    "Pakistan Today",
    "Pakistan Press International",
    "Pakistan Official News",
    "Qatar Tribune",
    "Gulf News (UAE)",
    "Gulf Times",

    # Other explicit org sources
    "COCAP-Nepal Monitor",
    "Radio Free Asia",
    "EFE (Spanish News Agency)",
}

# =========================
# NORMALIZATION + TOKENIZATION
# =========================
def normalize(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

NATIONAL_NEWSPAPERS = {normalize(x) for x in NATIONAL_NEWSPAPERS}
NON_NEWSPAPER_SOURCES = {normalize(x) for x in NON_NEWSPAPER_SOURCES}

def tokenize_sources(source_str):
    """
    ACLED 'source' can contain multiple sources per event.
    We split on ';' first, then ',' and normalize whitespace.
    """
    if pd.isna(source_str):
        return []
    parts = []
    for chunk in str(source_str).split(";"):
        for sub in chunk.split(","):
            t = sub.strip()
            if t:
                parts.append(normalize(t))
    return parts

# =========================
# CLASSIFICATION (EXACTLY WHAT WE DO)
# =========================
# For each event:
# - if it has any national newspaper source -> national
# - if it has any regional/local source (not national, not non-newspaper) -> regional/local
# - if it has both -> mixed
# - if it has neither (only agencies/NGOs/broadcasters etc.) -> other/no_match
def classify_event_sources(source_str):
    sources = tokenize_sources(source_str)

    has_national = any(s in NATIONAL_NEWSPAPERS for s in sources)
    has_non_news = any(s in NON_NEWSPAPER_SOURCES for s in sources)
    has_regional = any(
        (s not in NATIONAL_NEWSPAPERS) and (s not in NON_NEWSPAPER_SOURCES)
        for s in sources
    )

    if has_national and has_regional:
        bucket_4way = "mixed_nat+regional"
    elif has_national:
        bucket_4way = "national_newspaper"
    elif has_regional:
        bucket_4way = "regional_local_newspaper"
    else:
        bucket_4way = "no_newspaper_match"

    # The 2-way split Nausheen asked for:
    # - treat "mixed" as national_incl_mixed (because a national newspaper reported it)
    # - compare against purely regional/local
    if bucket_4way in ("national_newspaper", "mixed_nat+regional"):
        bucket_2way = "national_incl_mixed"
    elif bucket_4way == "regional_local_newspaper":
        bucket_2way = "regional_local"
    else:
        bucket_2way = "other"

    return pd.Series({
        "has_national": has_national,
        "has_regional": has_regional,
        "has_non_news": has_non_news,
        "bucket_4way": bucket_4way,
        "bucket_2way": bucket_2way
    })

# =========================
# LOAD ACLED + FILTER YEARS
# =========================
df = pd.read_csv(ACLED_CSV_PATH)

required_cols = {"event_date", "source", "event_type", "fatalities", "admin1", "admin2", "location"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns in ACLED file: {missing}")

df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df = df[(df["event_date"] >= pd.to_datetime(DATE_MIN)) & (df["event_date"] <= pd.to_datetime(DATE_MAX))].copy()
df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").fillna(0)

print("Rows (filtered):", len(df))
print("Date range:", df["event_date"].min(), "->", df["event_date"].max())
print("Total fatalities:", int(df["fatalities"].sum()))

# =========================
# APPLY CLASSIFICATION
# =========================
cls = df["source"].apply(classify_event_sources)
df = pd.concat([df, cls], axis=1)

print("\nBucket counts (4-way):")
print(df["bucket_4way"].value_counts())
print("\nBucket counts (2-way + other):")
print(df["bucket_2way"].value_counts())

# Save classified dataset for verification
df.to_csv(OUTDIR / "acled_with_source_buckets_2016_2024.csv", index=False)

# =========================
# EDA (National vs Regional/Local) — EXACT COMPARISON SETS
# =========================
sub = df[df["bucket_2way"].isin(["national_incl_mixed", "regional_local"])].copy()

def summary_table(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = (data.groupby(group_col)
             .agg(
                 events=("event_date", "size"),
                 fatalities=("fatalities", "sum"),
                 mean_fatalities=("fatalities", "mean"),
                 median_fatalities=("fatalities", "median"),
                 admin1_units=("admin1", "nunique"),     # states + UTs in India (=36)
                 admin2_units=("admin2", "nunique"),
                 locations=("location", "nunique"),
             )
             .reset_index()
             .sort_values("events", ascending=False))
    out["fatalities"] = out["fatalities"].astype(int)
    return out

summary_2way = summary_table(sub, "bucket_2way")
summary_2way.to_csv(OUTDIR / "eda_summary_national_vs_regional.csv", index=False)

# Event-type composition (% share within each subset)
event_type_shares = (pd.crosstab(sub["event_type"], sub["bucket_2way"], normalize="columns") * 100).round(2)
event_type_shares.to_csv(OUTDIR / "eda_event_type_shares_national_vs_regional.csv")

# Geographic coverage by admin1 (how many events per admin1 in each subset)
admin1_counts = sub.groupby(["admin1", "bucket_2way"]).size().reset_index(name="events")
admin1_counts.to_csv(OUTDIR / "eda_admin1_event_counts_by_subset.csv", index=False)

# Admin1 skew: share(national) - share(regional)
admin1_share = pd.crosstab(sub["admin1"], sub["bucket_2way"], normalize="columns")
admin1_share["diff_nat_minus_reg"] = admin1_share["national_incl_mixed"] - admin1_share["regional_local"]
admin1_share_sorted = admin1_share.sort_values("diff_nat_minus_reg", ascending=False)
admin1_share_sorted.reset_index().to_csv(OUTDIR / "eda_admin1_skew_national_minus_regional.csv", index=False)

# Trends over time (events + fatalities by year)
sub["year"] = sub["event_date"].dt.year
yearly = (sub.groupby(["year", "bucket_2way"])
            .agg(events=("event_date", "size"), fatalities=("fatalities", "sum"))
            .reset_index())
yearly.to_csv(OUTDIR / "eda_yearly_events_fatalities_by_subset.csv", index=False)

# Severity proxy: fatalities per 100 events by event type
sev = (sub.groupby(["bucket_2way", "event_type"])
         .agg(events=("event_date", "size"), fatalities=("fatalities", "sum"))
         .reset_index())
sev["fatalities_per_100_events"] = (sev["fatalities"] / sev["events"] * 100).replace([np.inf, -np.inf], np.nan)
sev.sort_values("fatalities_per_100_events", ascending=False).to_csv(
    OUTDIR / "eda_severity_fatalities_per_100_events_by_event_type.csv", index=False
)

# =========================
# PLOTS (EXPORT AS PNG)
# =========================
def bar_plot(x, y, title, xlabel, ylabel, outpath, rotate=0):
    plt.figure(figsize=(9, 5))
    plt.bar(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if rotate:
        plt.xticks(rotation=rotate, ha="right")
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# Events & fatalities bars
tmp = summary_2way.set_index("bucket_2way")
bar_plot(
    tmp.index.tolist(), tmp["events"].tolist(),
    "ACLED: Events (National vs Regional/Local)",
    "Subset", "Events",
    OUTDIR / "plot_events_national_vs_regional.png"
)
bar_plot(
    tmp.index.tolist(), tmp["fatalities"].tolist(),
    "ACLED: Fatalities (National vs Regional/Local)",
    "Subset", "Fatalities",
    OUTDIR / "plot_fatalities_national_vs_regional.png"
)

# Event-type composition side-by-side
etype = event_type_shares.copy()
etype = etype.loc[etype.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(10, 6))
x = np.arange(len(etype.index))
w = 0.4
plt.bar(x - w/2, etype["national_incl_mixed"], width=w, label="national_incl_mixed")
plt.bar(x + w/2, etype["regional_local"], width=w, label="regional_local")
plt.xticks(x, etype.index, rotation=30, ha="right")
plt.ylabel("Share of events (%)")
plt.title("ACLED: Event-type composition (National vs Regional/Local)")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "plot_event_type_composition_national_vs_regional.png", dpi=200)
plt.close()

# Admin1 skew plots (top 15 national-heavy & top 15 regional-heavy)
diff = admin1_share_sorted["diff_nat_minus_reg"]
top_nat = diff.head(15)
top_reg = diff.tail(15)

plt.figure(figsize=(10, 6))
plt.bar(top_nat.index, top_nat.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share difference (national - regional)")
plt.title("Admin1 units more represented in NATIONAL subset (top 15)")
plt.tight_layout()
plt.savefig(OUTDIR / "plot_admin1_skew_more_national.png", dpi=200)
plt.close()

plt.figure(figsize=(10, 6))
plt.bar(top_reg.index, top_reg.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share difference (national - regional)")
plt.title("Admin1 units more represented in REGIONAL/LOCAL subset (top 15)")
plt.tight_layout()
plt.savefig(OUTDIR / "plot_admin1_skew_more_regional.png", dpi=200)
plt.close()

# Yearly trends lines
pivot_events = yearly.pivot(index="year", columns="bucket_2way", values="events").fillna(0)
plt.figure(figsize=(10, 6))
for col in pivot_events.columns:
    plt.plot(pivot_events.index, pivot_events[col], marker="o", label=col)
plt.title("ACLED: Events over time (National vs Regional/Local)")
plt.xlabel("Year")
plt.ylabel("Events")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "plot_yearly_events_national_vs_regional.png", dpi=200)
plt.close()

pivot_fat = yearly.pivot(index="year", columns="bucket_2way", values="fatalities").fillna(0)
plt.figure(figsize=(10, 6))
for col in pivot_fat.columns:
    plt.plot(pivot_fat.index, pivot_fat[col], marker="o", label=col)
plt.title("ACLED: Fatalities over time (National vs Regional/Local)")
plt.xlabel("Year")
plt.ylabel("Fatalities")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "plot_yearly_fatalities_national_vs_regional.png", dpi=200)
plt.close()

# =========================
# EXPORT MANIFEST + ZIP
# =========================
manifest = pd.DataFrame(
    [{"file": str(p), "bytes": p.stat().st_size}
     for p in sorted(OUTDIR.glob("**/*")) if p.is_file()]
)
manifest.to_csv(OUTDIR / "MANIFEST.csv", index=False)

zip_path = Path("acled_source_split_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTDIR.glob("**/*")):
        if p.is_file():
            z.write(p, arcname=str(p))

# =========================
# PRINT QUICK RESULTS
# =========================
print("\n✅ Done.")
print("Outputs folder:", OUTDIR.resolve())
print("ZIP file:", zip_path.resolve())
print("\nSummary (National vs Regional/Local):")
print(summary_2way)


Rows (filtered): 174070
Date range: 2016-01-01 00:00:00 -> 2024-12-27 00:00:00
Total fatalities: 12353

Bucket counts (4-way):
bucket_4way
regional_local_newspaper    93677
national_newspaper          65545
no_newspaper_match           9255
mixed_nat+regional           5593
Name: count, dtype: int64

Bucket counts (2-way + other):
bucket_2way
regional_local         93677
national_incl_mixed    71138
other                   9255
Name: count, dtype: int64

✅ Done.
Outputs folder: /home/aniru/UCI/outputs
ZIP file: /home/aniru/UCI/acled_source_split_outputs.zip

Summary (National vs Regional/Local):
           bucket_2way  events  fatalities  mean_fatalities  \
1       regional_local   93677        5289         0.056460   
0  national_incl_mixed   71138        5063         0.071172   

   median_fatalities  admin1_units  admin2_units  locations  
1                0.0            36           727      10851  
0                0.0            36           709       9079  
